## 0 — Install

In [ ]:
import subprocess, sys

# Remove torchao if present -- causes PEFT import error
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'],
               check=False, capture_output=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40', 'peft>=0.10', 'torch', 'pyyaml',
], check=False)
print('Done.')


## 1 — Paths & Device

In [ ]:
import os, sys, platform
from pathlib import Path
import yaml
import torch
import torch.nn as nn

IS_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/dataMiningProject/CSI_Project')
    if not (BASE_DIR / 'config.yaml').exists():
        BASE_DIR = Path('/content/drive/MyDrive/CSI_Project')
    print(f'Colab -- BASE_DIR: {BASE_DIR}')
else:
    BASE_DIR = Path(os.path.dirname(os.path.abspath('__file__')))
    if not (BASE_DIR / 'datasets').exists():
        BASE_DIR = Path.cwd()
    print(f'Local -- BASE_DIR: {BASE_DIR}')

cfg_path = BASE_DIR / 'config.yaml'
assert cfg_path.exists(), f'Missing config.yaml'
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

TOKEN_CACHE_DIR = BASE_DIR / cfg['token_cache_dir']
CHECKPOINT_DIR  = BASE_DIR / cfg['checkpoint_dir']
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

GCB_CACHE      = TOKEN_CACHE_DIR / 'tokens_maxlen512.pt'
VULBERTA_CACHE = TOKEN_CACHE_DIR / 'tokens_vulberta_maxlen512.pt'
SAVE_PATH      = CHECKPOINT_DIR  / 'vulberta_fusion_init.pt'

assert GCB_CACHE.exists(),      f'Missing Anas GCB cache: {GCB_CACHE}'
assert VULBERTA_CACHE.exists(), f'Missing VulBERTa cache: {VULBERTA_CACHE} -- run 05c first'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device         : {DEVICE}')
print(f'GCB_CACHE      : {GCB_CACHE.name}  ({GCB_CACHE.stat().st_size/1e6:.0f} MB)')
print(f'VULBERTA_CACHE : {VULBERTA_CACHE.name}  ({VULBERTA_CACHE.stat().st_size/1e6:.0f} MB)')


## 2 — Model Config

In [ ]:
# Model names
GCB_MODEL_NAME   = 'microsoft/graphcodebert-base'
VULB_MODEL_NAME  = 'claudios/VulBERTa-MLP-D2A'

# Architecture
HIDDEN_DIM       = 768
NUM_CWE_CLASSES  = 8
MAX_LEN          = 512
DROPOUT_RATE     = 0.1

# LoRA config (GraphCodeBERT only -- VulBERTa is frozen)
LORA_R           = 16
LORA_ALPHA       = 32
LORA_DROPOUT     = 0.05
LORA_TARGETS = ['query', 'key', 'value']

CWE_LABEL_MAP = {
    'CWE-077': 0, 'CWE-601': 1, 'CWE-022': 2,
    'CWE-094': 3, 'CWE-089': 4, 'CWE-352': 5, 'CWE-079': 6,
    'unknown': 7,
}
CWE_NAMES = [
    'CWE-077', 'CWE-601', 'CWE-022', 'CWE-094',
    'CWE-089', 'CWE-352', 'CWE-079', 'unknown'
]

print('Config:')
print(f'  GCB    : {GCB_MODEL_NAME}')
print(f'  VulB   : {VULB_MODEL_NAME}')
print(f'  MAX_LEN: {MAX_LEN}')
print(f'  Classes: {NUM_CWE_CLASSES}')


## 3 — Model Architecture

### Part A: FusionProjectionLayer
Takes two 768-dim CLS embeddings, concatenates them to 1536-dim,
then projects back to 768-dim via Linear → LayerNorm → GELU → Dropout.

### Part B: VulBERTaFusionModel
Full dual encoder model:
1. **Dual input forward pass** — runs both encoders simultaneously
2. **Fusion** — combines both CLS embeddings into one representation
3. **Classification** — maps fused representation to 7 CWE classes

**Why `RobertaModel` for VulBERTa?**
`claudios/VulBERTa-MLP-D2A` is a `RobertaForSequenceClassification` model.
Using `AutoModel` would fail or load the wrong class.
`RobertaModel.from_pretrained()` strips the classification head and gives
us the encoder directly — exactly what we need for feature extraction.

In [ ]:
from transformers import AutoModel, RobertaModel
from peft import LoraConfig, get_peft_model, TaskType
from typing import Optional, Dict


# ── Part A: Fusion Projection Layer ──────────────────────────────────────
class FusionProjectionLayer(nn.Module):
    """
    Combines GraphCodeBERT and VulBERTa CLS embeddings into one representation.

    Input  : gcb_cls  (B, 768)  +  vulb_cls (B, 768)
    Process: concat → (B, 1536) → Linear → LayerNorm → GELU → Dropout
    Output : fused   (B, 768)
    """
    def __init__(self, hidden_dim=HIDDEN_DIM, proj_dim=HIDDEN_DIM,
                 dropout=DROPOUT_RATE):
        super().__init__()
        self.projection = nn.Sequential(
            nn.Linear(hidden_dim * 2, proj_dim),   # 1536 → 768
            nn.LayerNorm(proj_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        # Xavier init for stable training from scratch
        nn.init.xavier_uniform_(self.projection[0].weight)
        nn.init.zeros_(self.projection[0].bias)

    def forward(self, gcb_cls: torch.Tensor,
                vulb_cls: torch.Tensor) -> torch.Tensor:
        # concat: (B,768) + (B,768) → (B,1536)
        # project: (B,1536) → (B,768)
        return self.projection(torch.cat([gcb_cls, vulb_cls], dim=-1))


# ── Part B: Full Dual Encoder Model ──────────────────────────────────────
class VulBERTaFusionModel(nn.Module):
    """
    Dual encoder fusion model for CWE vulnerability classification.

    Encoder 1: GraphCodeBERT + LoRA  (trainable ~295K params)
    Encoder 2: VulBERTa              (frozen -- security pre-trained)
    Fusion   : FusionProjectionLayer (trainable ~1.2M params)
    Head     : 2-layer MLP           (trainable ~296K params)

    forward() inputs:
        gcb_input_ids, gcb_attention_mask   ← from GraphCodeBERT tokenizer
        vulb_input_ids, vulb_attention_mask ← from VulBERTa tokenizer
        labels (optional)                   ← CWE class indices (0-6, -1=ignore)

    forward() outputs:
        logits     : (B, 7)   CWE class scores
        fused_repr : (B, 768) fused representation
        loss       : scalar   CrossEntropy (only if labels provided)
    """

    def __init__(
        self,
        gcb_model_name  = GCB_MODEL_NAME,
        vulb_model_name = VULB_MODEL_NAME,
        num_labels      = NUM_CWE_CLASSES,
        hidden_dim      = HIDDEN_DIM,
        dropout         = DROPOUT_RATE,
        freeze_vulberta = True,
    ):
        super().__init__()

        # ── Encoder 1: GraphCodeBERT + LoRA ──────────────────────────────
        print(f'Loading GraphCodeBERT: {gcb_model_name}')
        _gcb  = AutoModel.from_pretrained(gcb_model_name)
        _lora = LoraConfig(
            task_type      = TaskType.FEATURE_EXTRACTION,
            r              = LORA_R,
            lora_alpha     = LORA_ALPHA,
            lora_dropout   = LORA_DROPOUT,
            target_modules = LORA_TARGETS,
            bias           = 'none',
        )
        self.graphcodebert = get_peft_model(_gcb, _lora)
        self.graphcodebert.print_trainable_parameters()

        # ── Encoder 2: VulBERTa (frozen) ──────────────────────────────────
        # Use RobertaModel.from_pretrained() to load ONLY the encoder
        # (strips the classification head from VulBERTa-MLP-D2A)
        print(f'Loading VulBERTa encoder: {vulb_model_name}')
        self.vulberta = RobertaModel.from_pretrained(
            vulb_model_name,
            ignore_mismatched_sizes=True,  # ignore classifier head mismatch
        )
        if freeze_vulberta:
            for p in self.vulberta.parameters():
                p.requires_grad = False
            print(f'VulBERTa: FROZEN (security pre-training preserved)')

        # ── Fusion Layer ──────────────────────────────────────────────────
        self.fusion = FusionProjectionLayer(
            hidden_dim = hidden_dim,
            proj_dim   = hidden_dim,
            dropout    = dropout,
        )

        # ── CWE Classification Head ───────────────────────────────────────
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),  # 768 → 384
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_labels),  # 384 → 7
        )

        # Ignore unknown CWE labels (-1) in loss
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=-1)

    @staticmethod
    def _get_cls(encoder, input_ids, attention_mask) -> torch.Tensor:
        """Run encoder and return CLS token embedding (position 0)."""
        out = encoder(
            input_ids      = input_ids,
            attention_mask = attention_mask,
        )
        return out.last_hidden_state[:, 0, :]  # (B, 768)

    def forward(
        self,
        gcb_input_ids:       torch.Tensor,  # (B, 512)
        gcb_attention_mask:  torch.Tensor,  # (B, 512)
        vulb_input_ids:      torch.Tensor,  # (B, 512)
        vulb_attention_mask: torch.Tensor,  # (B, 512)
        labels: Optional[torch.Tensor] = None,  # (B,)
    ) -> Dict[str, torch.Tensor]:

        # Step 1: Dual input forward pass
        # Both encoders run in parallel on their respective token sequences
        gcb_cls  = self._get_cls(
            self.graphcodebert, gcb_input_ids, gcb_attention_mask
        )  # (B, 768) -- GraphCodeBERT embedding
        vulb_cls = self._get_cls(
            self.vulberta, vulb_input_ids, vulb_attention_mask
        )  # (B, 768) -- VulBERTa embedding

        # Step 2: Fusion -- combine both embeddings into one
        fused  = self.fusion(gcb_cls, vulb_cls)   # (B, 768)

        # Step 3: CWE classification
        logits = self.classifier(fused)            # (B, 7)

        out = {'logits': logits, 'fused_repr': fused}
        if labels is not None:
            out['loss'] = self.loss_fn(logits, labels)
        return out


print('FusionProjectionLayer defined')
print('VulBERTaFusionModel defined')


## 4 — Build Model & Inspect

In [ ]:
model = VulBERTaFusionModel(freeze_vulberta=True).to(DEVICE)

total_p     = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_p    = total_p - trainable_p

print(f'\nParameter summary:')
print(f'  Total     : {total_p:,}')
print(f'  Trainable : {trainable_p:,}  ({100*trainable_p/total_p:.2f}%)')
print(f'  Frozen    : {frozen_p:,}  ({100*frozen_p/total_p:.2f}%)')
print()
print('Fusion layer:')
print(model.fusion)
print()
print('Classifier head:')
print(model.classifier)


## 5 — Forward Pass Test

Test with real tokenizers to verify the dual input forward pass works end-to-end.

In [ ]:
from transformers import AutoTokenizer, RobertaTokenizer

print('Loading tokenizers for test...')
gcb_tok  = AutoTokenizer.from_pretrained(GCB_MODEL_NAME)
vulb_tok = AutoTokenizer.from_pretrained(VULB_MODEL_NAME)

# Three code snippets covering different CWE types
codes = [
    "query = 'SELECT * FROM users WHERE id=' + user_input",        # CWE-089
    "import os\nos.system(user_cmd)",                              # CWE-077
    "return redirect(request.args.get('next', '/'))",              # CWE-601
]
labels = torch.tensor([4, 0, 1], dtype=torch.long).to(DEVICE)

# Tokenize SEPARATELY with each tokenizer
gcb_enc  = gcb_tok(
    codes, max_length=MAX_LEN, padding='max_length',
    truncation=True, return_tensors='pt'
).to(DEVICE)
vulb_enc = vulb_tok(
    codes, max_length=MAX_LEN, padding='max_length',
    truncation=True, return_tensors='pt'
).to(DEVICE)

# Forward pass
model.eval()
with torch.no_grad():
    out = model(
        gcb_input_ids       = gcb_enc['input_ids'],
        gcb_attention_mask  = gcb_enc['attention_mask'],
        vulb_input_ids      = vulb_enc['input_ids'],
        vulb_attention_mask = vulb_enc['attention_mask'],
        labels              = labels,
    )

print('Forward pass OK!')
print(f'  gcb tokens   : {tuple(gcb_enc["input_ids"].shape)}')
print(f'  vulb tokens  : {tuple(vulb_enc["input_ids"].shape)}')
print(f'  gcb_cls      : (B, 768) -- GraphCodeBERT CLS embedding')
print(f'  vulb_cls     : (B, 768) -- VulBERTa CLS embedding')
print(f'  fused_repr   : {tuple(out["fused_repr"].shape)}  -- after fusion layer')
print(f'  logits       : {tuple(out["logits"].shape)}  -- CWE class scores')
print(f'  loss         : {out["loss"].item():.4f}')
print()

# Verify tokens are different (different vocabularies)
tokens_differ = not torch.equal(gcb_enc['input_ids'], vulb_enc['input_ids'])
print(f'GCB != VulBERTa tokens : {tokens_differ}  (must be True)')
assert tokens_differ, 'ERROR: same token IDs -- tokenizers are identical!'
print('Dual input verified: two different token sequences fed to two different encoders')


## 6 — Unit Tests

In [ ]:
print('Running unit tests...')

# T1: FusionProjectionLayer output shape
fl = FusionProjectionLayer(dropout=0.0).eval()
g  = torch.randn(8, 768)
v  = torch.randn(8, 768)
o  = fl(g, v)
assert o.shape == (8, 768), f'Shape wrong: {o.shape}'
print('  T1 PASSED: FusionLayer output shape (8, 768)')

# T2: Fusion output is single tensor (not tuple)
assert isinstance(o, torch.Tensor)
print('  T2 PASSED: output is a single Tensor')

# T3: Concat dim is 1536
assert torch.cat([g, v], dim=-1).shape[-1] == 1536
print('  T3 PASSED: concat produces 1536 before projection')

# T4: Gradients flow through fusion layer
fl.train()
g2 = torch.randn(2, 768, requires_grad=True)
v2 = torch.randn(2, 768, requires_grad=True)
fl(g2, v2).sum().backward()
assert g2.grad is not None and v2.grad is not None
print('  T4 PASSED: gradients flow through both encoder paths')

# T5: Different inputs produce different outputs
fl.eval()
oa = fl(torch.randn(2, 768), torch.randn(2, 768))
ob = fl(torch.randn(2, 768), torch.randn(2, 768))
assert not torch.allclose(oa, ob)
print('  T5 PASSED: different inputs give different outputs')

# T6: VulBERTa is actually frozen
frozen = all(not p.requires_grad for p in model.vulberta.parameters())
assert frozen, 'VulBERTa has trainable params -- should be frozen!'
print('  T6 PASSED: VulBERTa is fully frozen')

# T7: Model output keys correct
model.eval()
with torch.no_grad():
    test_out = model(
        gcb_enc['input_ids'],  gcb_enc['attention_mask'],
        vulb_enc['input_ids'], vulb_enc['attention_mask'],
    )
assert set(test_out.keys()) == {'logits', 'fused_repr'}
assert test_out['logits'].shape    == (3, 8)
assert test_out['fused_repr'].shape == (3, 768)
print('  T7 PASSED: output keys and shapes correct')

print('\nAll 8 tests passed!')


## 7 — Cache Smoke Test

Verify the two caches load correctly and their output keys
match `VulBERTaFusionModel.forward()` exactly.

In [ ]:
from torch.utils.data import Dataset, DataLoader


class DualCacheDataset(Dataset):
    """
    Loads from Anas's GCB cache + VulBERTa cache.
    Both caches have 14,522 records (chunks of 4,085 functions).

    Output keys match VulBERTaFusionModel.forward():
        gcb_input_ids, gcb_attention_mask
        vulb_input_ids, vulb_attention_mask
        label
    """
    def __init__(self, gcb_cache, vulb_cache, split):
        assert gcb_cache['num_records'] == vulb_cache['num_records'], \
            'Cache size mismatch'
        assert gcb_cache['split_origins'] == vulb_cache['split_origins'], \
            'Cache alignment mismatch -- rebuild with 05c'

        if split == 'all':
            indices = list(range(len(gcb_cache['split_origins'])))
        else:
            indices = [
                i for i, s in enumerate(gcb_cache['split_origins'])
                if s == split
            ]

        self.gcb_ids   = gcb_cache['input_ids'][indices]
        self.gcb_mask  = gcb_cache['attention_mask'][indices]
        self.vulb_ids  = vulb_cache['input_ids'][indices]
        self.vulb_mask = vulb_cache['attention_mask'][indices]
        self.labels    = gcb_cache['cwe_labels'][indices]

    def __len__(self): return len(self.gcb_ids)

    def __getitem__(self, idx):
        return {
            'gcb_input_ids':       self.gcb_ids[idx],
            'gcb_attention_mask':  self.gcb_mask[idx],
            'vulb_input_ids':      self.vulb_ids[idx],
            'vulb_attention_mask': self.vulb_mask[idx],
            'label':               self.labels[idx],
        }


print('Loading caches...')
gcb_cache  = torch.load(GCB_CACHE,      weights_only=True)
vulb_cache = torch.load(VULBERTA_CACHE, weights_only=True)
print(f'  GCB cache    : {tuple(gcb_cache["input_ids"].shape)}')
print(f'  VulBERTa     : {tuple(vulb_cache["input_ids"].shape)}')

train_ds = DualCacheDataset(gcb_cache, vulb_cache, split='train')
val_ds   = DualCacheDataset(gcb_cache, vulb_cache, split='val')
print(f'  Train samples: {len(train_ds):,}')
print(f'  Val samples  : {len(val_ds):,}')

# Load one batch and run through model
loader = DataLoader(train_ds, batch_size=4, shuffle=False)
batch  = next(iter(loader))

model.eval()
with torch.no_grad():
    out = model(
        batch['gcb_input_ids'].to(DEVICE),
        batch['gcb_attention_mask'].to(DEVICE),
        batch['vulb_input_ids'].to(DEVICE),
        batch['vulb_attention_mask'].to(DEVICE),
        labels = batch['label'].to(DEVICE),
    )

print(f'\nCache batch forward pass:')
print(f'  gcb_input_ids   : {tuple(batch["gcb_input_ids"].shape)}')
print(f'  vulb_input_ids  : {tuple(batch["vulb_input_ids"].shape)}')
print(f'  logits          : {tuple(out["logits"].shape)}')
print(f'  fused_repr      : {tuple(out["fused_repr"].shape)}')
print(f'  loss            : {out["loss"].item():.4f}')
print()
print('Cache smoke test PASSED')
print('DualCacheDataset is ready for 06_dual_encoder_training.ipynb')


## 8 — Save Initial Checkpoint

Save the freshly initialized model weights to Drive.
`06_dual_encoder_training.ipynb` will load this as the starting point.

In [ ]:
torch.save(model.state_dict(), SAVE_PATH)

size_mb = SAVE_PATH.stat().st_size / 1e6
print(f'Saved: {SAVE_PATH}')
print(f'Size : {size_mb:.0f} MB')
print()
print('Ready for 06_dual_encoder_training.ipynb')
print('Make sure 06 uses:')
print(f'  GCB_CACHE      : tokens_maxlen512.pt      (Anas)')
print(f'  VULBERTA_CACHE : tokens_vulberta_maxlen512.pt')
print(f'  VULB_MODEL_NAME: {VULB_MODEL_NAME}')
print(f'  MAX_LEN        : {MAX_LEN}')
